In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

root = "logs"
fig_save_folder = os.path.join("figs", "sanity_check")

# Initial progress on Limited Tomography

In `2026_09_13/exp_0`, we ran just one seed of a simple sanity check for:
- `exp_0`: TRIANGLE reward, n_angles=6, and E2E (entropy_coeff=0.0)
- `exp_1`: TRIANGLE reward, n_angles=24, and E2E (entropy_coeff=0.0)
- `exp_2`: TRIANGLE reward, n_angles=96, and E2E (entropy_coeff=0.0)
- `exp_3`: TRIANGLE reward, n_angles=6, and INC (entropy_coeff=0.0)
- `exp_4`: TRIANGLE reward, n_angles=24, and INC (entropy_coeff=0.0)
- `exp_5`: TRIANGLE reward, n_angles=96, and INC (entropy_coeff=0.0)
- `exp_6`: TRIANGLE_FIXED reward, n_angles=6, and E2E (entropy_coeff=0.0)
- `exp_7`: TRIANGLE_FIXED reward, n_angles=24, and E2E (entropy_coeff=0.0)
- `exp_8`: TRIANGLE_FIXED reward, n_angles=96, and E2E (entropy_coeff=0.0)
- `exp_9`: TRIANGLE_FIXED reward, n_angles=6, and INC (entropy_coeff=0.0)
- `exp_10`: TRIANGLE_FIXED reward, n_angles=24, and INC (entropy_coeff=0.0)
- `exp_11`: TRIANGLE_FIXED reward, n_angles=96, and INC (entropy_coeff=0.0)

In [ ]:
max_ep = 30_000
n_seeds = 1
seed_offset = 1029
n_exps = 12
save_figs = True

In [ ]:
n_eps_arr = np.zeros((n_exps, n_seeds), dtype=int)
episode = np.zeros((n_exps, n_seeds, max_ep), dtype=int)
reward = np.zeros((n_exps, n_seeds, max_ep), dtype=float)
entropy = np.zeros((n_exps, n_seeds, max_ep), dtype=float)
l_1_dist = np.zeros((n_exps, n_seeds, max_ep), dtype=float)
exp_id = 0

for i in range(n_exps):
    for k in range(n_seeds):
        j = seed_offset + k
        fname = os.path.join(root, "2026_09_13", "exp_0", "run_%d" % i, "seed=%d.csv" % j)
        df = pd.read_csv(fname, header="infer")
    
        n_eps_arr[i,k] = len(df)
        episode[i,k,:n_eps_arr[i,k]] = df['episode']
        reward[i,k,:n_eps_arr[i,k]] = df['episodic reward']
        entropy[i,k,:n_eps_arr[i,k]] = df['entropy']
        l_1_dist[i,k,:n_eps_arr[i,k]] = df['l_1']

### Plot score

In [ ]:
plt.style.use('ggplot')
_, axes = plt.subplots(ncols=2, nrows=2, figsize=(7,8))
title_arr = ["TRI_360deg", "TRI_1deg"]
label_arr = ["E2E", "INCREMENT"]
sub_label_arr = [6, 24, 96]
k = 0 # seed
avg_len = 10
avg_arr = np.ones(avg_len, dtype=float)/avg_len

for i in range(n_exps):
    i_col = (i//3) % 2
    i_row = i//6
    xs = episode[i,k,:n_eps_arr[i,k]][avg_len-1:-(avg_len-1)]
    ys = np.convolve(reward[i,k,:n_eps_arr[i,k]] , avg_arr, mode="same")[avg_len-1:-(avg_len-1)]
    axes[i_row, i_col].plot(xs, ys, label="%d scans" % sub_label_arr[i%3])

axes[0,0].legend()
for i in range(2):
    axes[i,0].set(
        xlim=(0,1_000),
        ylabel="Reward smoothed(10) (higher is better)",
        xlabel="Episodes",
        title="%s %s reward" % (title_arr[i], label_arr[0]),
    )
    axes[i,1].set(
        xlim=(0,2_000),
        xlabel="Episodes",
        title="%s %s reward" % (title_arr[i], label_arr[1]),
    )

plt.tight_layout()
if save_figs:
    fname = os.path.join(fig_save_folder, "reward.png")
    plt.savefig(fname, dpi=180)

**Experimental Results**:

- 96 scans stopped early because it takes many more samples/observations per episode, and all methods had a 30m time limit.
- `6 scans` starts later at around iteration 40 because 1) it had more logs, and our code stores on a subset of iterations (multiples of say 10). Then when we smooth it, we remove the first few log entries, which is why it appears to start much later.

### Plot distance to uniform
We will measure entropy of policies encountered during the algorithm.

$$
H(p) := -\sum_{a \in \mathrm{angles}} p(a) \ln(a).
$$

The higher $H(\cdot)$, the closer we are to a uniform policy.

In [ ]:
plt.style.use('ggplot')
_, axes = plt.subplots(ncols=2, nrows=2, figsize=(7,8))
title_arr = ["TRI_360deg", "TRI_1deg"]
label_arr = ["E2E", "INCREMENT"]
sub_label_arr = [6, 24, 96]
k = 0 # seed

for i in range(n_exps):
    i_col = (i//3) % 2
    i_row = i//6
    xs = episode[i,k,:n_eps_arr[i,k]]
    ys = entropy[i,k,:n_eps_arr[i,k]]
    axes[i_row, i_col].plot(xs, ys, label="%d scans" % sub_label_arr[i%3])

axes[0,0].legend()
for i in range(2):
    axes[i,0].set(
        xlim=(0,1_000),
        ylabel="Entropy (higher is closer to Uni)",
        xlabel="Episodes",
        title="%s %s reward" % (title_arr[i], label_arr[0]),
    )
    axes[i,1].set(
        xlim=(0,2_000),
        xlabel="Episodes",
        title="%s %s entropy" % (title_arr[i], label_arr[1]),
    )

plt.tight_layout()
if save_figs:
    fname = os.path.join(fig_save_folder, "entropy.png")
    plt.savefig(fname, dpi=180)

### Best Angle

For each of the four problems, we will output the best set of angles with 24 scans.

In [ ]:
title_arr = ["TRI_360deg", "TRI_1deg  "]
label_arr = ["E2E", "INC"]
seed = seed_offset
top_n_angles = 2

for i in range(2):
    for j in range(2):
        k = 2*i+j
        exp_id = 1 + 3*k
        fname = os.path.join(root, "2026_09_13", "exp_0", "run_%d" % exp_id, "finaL_angle_seed=%d.csv" % seed)
        df = pd.read_csv(fname, header="infer")

        angles = df.iloc[0].to_numpy().copy()
        # radians -> degrees
        angles *= 360/(2*np.pi)
        angles = (100*angles).astype('int')/100
        angle_dist = df.iloc[1].to_numpy()

        angles_dist_sorted_idx = np.argsort(angle_dist, descending=True)

        print("%s %s: Best %d angles: %s (prob: %s)" % (
                title_arr[i], 
                label_arr[j], 
                top_n_angles,
                angles[angles_dist_sorted_idx[:top_n_angles]],
                angle_dist[angles_dist_sorted_idx[:top_n_angles]],
        ))